# 04 · Does the method preserve more motion at the same repair quality?

**This is the decision notebook.** It compares locked operating points
and asks whether the gain survives stronger baselines, uncertainty and
held-out conditions. Default evaluation uses development cases only.

Run each notebook in a fresh kernel, in order **00 → 04**. Notebook 05 is
an external visual stress test. These are offline restoration experiments:
the model may inspect the declared complete clip. They do not claim causal
forecasting or clinical diagnosis.

**The default is real data.** Export `MP_RUN_ROOT`, the AMASS and GAVD data
paths, and the model configuration before opening Jupyter. See the
[launch guide](../../slurm/motion-preservation/README.md).
For a CPU walkthrough of the mechanics, explicitly choose `MP_MODE=demo`
and a separate run directory. Demo outputs cannot establish a research result.

[Proposal](../../docs/studies/motion-preservation/protocol/proposal.md)
· [Notebook guide](README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

project_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(project_override).expanduser()] if project_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)  # Resolve manifest/config paths from the checkout in every kernel.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, Video, display
from gavd6_sjepa.research_directions.motion_preservation import workflow, plots

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
cfg = workflow.config_from_environment()
RUN_ROOT = Path(cfg.run_root)
print(f"Mode: {cfg.mode}; run directory: {RUN_ROOT}")
if cfg.mode == "demo":
    display(Markdown("**DEMO ONLY: generated fixtures and stand-in models. "
                     "These outputs are not evidence about AMASS, GAVD, or a pretrained prior.**"))

## 1. Choose the evaluation boundary explicitly

`MP_EVALUATION_SPLIT=development` gives the 48-hour pilot decision.
`MP_EVALUATION_SPLIT=final` opens the reserved event family and people.
Final mode builds and caches those cases only after the gate and
operating points exist. It never calls training or calibration.

The implemented final setting combines a new event family, camera angle
and corruption mechanism. It measures combined stress robustness, not
isolated event generalization. Separate matched nuisance ablations are
needed to attribute an effect to one of these changes.

If you change the method after seeing the final result, that result is
development evidence and a new independent final set is needed.

In [ ]:
SPLIT = os.environ.get("MP_EVALUATION_SPLIT", "development")
if SPLIT not in {"development", "final"}:
    raise ValueError("MP_EVALUATION_SPLIT must be development or final.")
print(f"Evaluation role: {SPLIT}")
if SPLIT == "final":
    workflow.require_fitted(cfg)
    print("Opening the reserved final cases with the saved gate and calibration.")
    final_cases = workflow.build_pairs(cfg, roles=("final",))
    final_cache = workflow.cache_predictions(cfg, roles=("final",))
    print(f"Final cases: {len(final_cases):,}; cached rows: {len(final_cache):,}")

## 2. Read the two axes correctly

For event descriptor `d`, clean motion `x` and edited truth `x_event`,
event magnitude is `a = d(x_event) - d(x)`. Retention for output `y` is

`1 - abs(d(y) - d(x_event)) / abs(a)`.

One means the event descriptor matches its reference. Zero means an error as
large as the event itself. Negative retention is allowed and must not
be clipped away. Overshoot and signed errors are reported separately.

Repair uses squared 3D error at observed joints, measured against the
correct truth for the case, including `x_event` when an event is present.
Noise removal of 0.25 means a quarter of that observed-joint squared
error was removed. The raw and repaired scores use the same mask.

Missing joints are interpolated before the prior runs. Their separate
`completion_mse_m2` score measures gap filling. Also inspect
`observed_mse_m2`, all-joint `mse_m2`, and `missing_fraction`. Missing
joints cannot inflate the primary repair ratio.

In [ ]:
# Worked metric example only. These numbers are not model results.
event_magnitude = 0.10
reference_descriptor = 0.10
candidate_descriptors = np.array([0.10, 0.07, 0.00, 0.23])
display(pd.DataFrame({"candidate_descriptor": candidate_descriptors,
                      "retention": 1 - np.abs(candidate_descriptors - reference_descriptor) / event_magnitude,
                      "signed_event_error": candidate_descriptors - reference_descriptor}))

## 3. Evaluate the saved operating points

Inspect achieved noise removal alongside retention. Calibration may not
transfer perfectly, so two methods can achieve different removal on new
people despite sharing a calibration target. Do not call a retention
gain a matched-quality improvement if repair quality is materially worse.
Both methods must reach the target on event-plus-noise cases, as well
as in the overall noisy-case average.

In [ ]:
started = perf_counter()
report = workflow.evaluate(cfg, split=SPLIT)
print(f"Evaluation completed in {perf_counter() - started:.1f} seconds.")
display(report["summary"])

In [ ]:
figure = plots.plot_tradeoff(report)
display(figure)
plt.close(figure)

## 4. Examine the cases behind the average

Check event-only and event-plus-noise groups separately. The latter is
the stronger test. Examine reliable and unreliable flow, held cameras,
missing observations and the observation-equivalent occlusion fixture.
An aggregate can hide a method that succeeds only when no repair is needed.

Confidence intervals should resample people with every trial and variant
kept together. If the manifest supplies only an uncertain group identity,
label the result accordingly. Repeated optimization seeds do not create
new independent people.

In [ ]:
scores = report["scores"]
display(scores.head(20))
grouping = [c for c in ("method", "fixture", "event_family", "event_present", "noise_present", "condition")
            if c in scores]
quantities = [c for c in ("retention", "noise_removal", "descriptor_abs_error", "descriptor_signed_error", "brier", "decided")
              if c in scores]
if grouping and quantities:
    display(scores.groupby(grouping, dropna=False)[quantities].mean())

## 5. Make the 48-hour decision

The proposal's provisional development target is at least **15 percentage
points more retention** than the strongest baseline, while removing at
least **25% of injected error**, with comparable achieved repair and a
person-grouped interval supporting improvement. These are practical
thresholds, not predicted effects or a promise of publication.

Stop the method claim if simple calibrated flow matches the learned
frontier, the gain vanishes after nuisance controls, or the prior never
meaningfully erases true events. Use a poor reference-informed mixture
diagnostic to reconsider the gate design, without calling it a certified
upper bound after projection.
Dropping an unhelpful optional V-JEPA feature branch does not invalidate
a useful flow-supported adapter.

In [ ]:
decision = report["decision"]
display(Markdown("```json\n" + json.dumps(decision, indent=2, default=str) + "\n```"))
if cfg.mode == "demo":
    print("Ignore empirical continue/stop interpretations: this run tests pipeline mechanics only.")

## What comes after a passing pilot?

Expand to three optimization seeds and more people, then evaluate the
reserved family once. A second prior requires a new set of predictions
for the same cases, matching the original feature convention. Keep the
learned gate and calibration fixed if claiming transfer across priors.
Do not call a smoother a second foundation prior.

Natural AMASS events and GAVD inspection test whether the synthetic
result transfers. A successful demo or synthetic-only result is not the
full proposed ICLR contribution.

Next: [05 · Inspect GAVD stress cases](05_gavd_visual_stress.ipynb).